# v3 Surrogate workshop

End-to-end walkthrough of the surrogate lifecycle used by v3 calibration:

1. Open the persisted LHS training bundle (`result.pkl`, `baseline.pkl`, `metadata.json`) and inspect **exactly what got saved**, especially the scale-factor ("knob") matrix.
2. Build the crosswalk-driven aggregation `A` and the training matrices `(X, Y)`.
3. Split into train / dev / test, train a `Surrogate`, evaluate it, apply the accuracy gate.
4. Persist and reload the resulting surrogate bundle.

**Assumed bundle:** the existing 1k-sample bundle at `sisepuede-surrogate-data/PER_2018_n1000_seed42/`. Every cell below runs against it. Point `BUNDLE_DIR` at a larger bundle (e.g. `PER_2018_N2000_seed42/`) to work with a real run.

**Caveats to keep in mind with `n_lhs=1000`: R$^2$ and MAPE are not really informative. Everything will be correct end-to-end but the numbers we see are for *plumbing verification*, not evidence.

## 0. Setup

In [ ]:
import json, os, sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "sisepuede").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# Bundle to inspect. Override with the real n=2000 bundle when it exists.
BUNDLE_DIR = Path(os.environ.get(
    "BUNDLE_DIR",
    # REPO_ROOT / "sisepuede-surrogate-data/PER_2018_n1000_seed42",
    "/Users/dianamendez/sisepuede-surrogate-data/PER_2018_n1000_seed42"
))
SURROGATE_OUT_DIR = Path(os.environ.get(
    "SURROGATE_OUT_DIR",
    REPO_ROOT / "sisepuede/out/surrogates",
))
print("REPO_ROOT        :", REPO_ROOT)
print("BUNDLE_DIR       :", BUNDLE_DIR, "(exists:", BUNDLE_DIR.exists(), ")")
print("SURROGATE_OUT_DIR:", SURROGATE_OUT_DIR)

## 1. What's inside the LHS training bundle

The training-data directory is produced by `generate_surrogate_data.py`. Layout:

```
{iso3}_{year}_n{N}_seed{S}[_tag]/
├── result.pkl        # pickled SensitivityResult (see below)
├── baseline.pkl      # pickled DataFrame: the post-v2 input frame used as the LHS anchor
└── metadata.json     # provenance: fingerprint, spec list, seed, wall-clock, git commit
```

**Nothing here is a CSV.** The bundles are pickle+JSON on purpose:

- `pd.DataFrame` survives round-trip with its dtypes intact (needed for `int` run indices, `object` code columns, and `float64` numeric columns).
- The three artifacts are kept aligned by shared identifiers (`run_index`, `year`, `iea_balance_code`, `iea_product_code`). CSV would lose the `pd.MultiIndex` structure used to reindex.
- If you want a CSV view of any one piece, `.to_csv()` from inside this notebook works.


In [ ]:
from sisepuede.calibration._data_generation import load_training_data

result, baseline, metadata = load_training_data(str(BUNDLE_DIR))
print("SensitivityResult attributes:")
for a in ["sampling_mode", "variable_specs", "input_samples",
          "iea_comparison", "model_outputs",
          "baseline_output", "baseline_iea_comparison"]:
    v = getattr(result, a)
    if hasattr(v, "shape"):
        print(f"  {a:26s} DataFrame  shape={v.shape}")
    elif isinstance(v, list):
        print(f"  {a:26s} list       len={len(v)}")
    else:
        print(f"  {a:26s} {type(v).__name__:10s} {v!r}")
print()
print("baseline.pkl (post-v2 input frame):", type(baseline).__name__, baseline.shape)

## 2. The scale-factor matrix (`result.input_samples`)

The LHS results are saved in **a plain `pd.DataFrame`**, of shape `(n_lhs, n_knobs)`:

- one row per LHS sample (index = `run_index`, 0…N-1),
- one column per calibration knob (`spec.column`),
- values are dimensionless scale factors sampled uniformly from `[knob_lb, knob_ub]` via LHS (i.e. multipliers applied to a SISEPUEDE input column at the target year).

This is what feeds the surrogate as `X`. It aligns row-for-row with `result.model_outputs` (via `run_index`) and with `result.iea_comparison`.


In [ ]:
X_raw = result.input_samples
print("type:", type(X_raw).__name__, " shape:", X_raw.shape)
print("index:", X_raw.index.tolist())
X_raw.head()

In [ ]:
# Per-column range check: min/max should land inside the LHS box
# recorded in metadata['knob_bounds'].
print("metadata['knob_bounds']:", metadata["knob_bounds"])
X_raw.describe().loc[["min", "max", "mean", "std"]].T.head(10)

In [ ]:
# Column names come straight from the VariableSpec list.
spec0 = result.variable_specs[0]
print("first VariableSpec :", spec0)
print("first column name  :", X_raw.columns[0])
print("spec.column matches:", spec0.column == X_raw.columns[0])
print("num knobs          :", len(result.variable_specs))

In [ ]:
# If we want a CSV of the scale factors specifically:
csv_out = BUNDLE_DIR / "input_samples_view.csv"
X_raw.to_csv(csv_out)
print("wrote", csv_out, "—", csv_out.stat().st_size, "bytes")

## 3. Baseline artifacts

Two related "baselines" live in the bundle. They mean different things:

| Attribute | Type | Meaning |
|---|---|---|
| `result.baseline_output` | DataFrame | Full SSP model output at the **unperturbed** input (all scale factors = 1.0). One row per `time_period`. |
| `result.baseline_iea_comparison` | DataFrame | The same run's IEA-crosswalked comparison table (used later for `baseline_targets(...)` and residual diagnostics). |
| `baseline.pkl` (top-level) | DataFrame | The **post-v2** SISEPUEDE input frame used as the LHS anchor. This is what the LHS multiplies by each `f` vector. |

The last one is the consumption state the surrogate is tied to via `metadata['consumption_fingerprint']`.


In [ ]:
print("baseline_output       :", result.baseline_output.shape)
print("baseline_iea_compar.  :", result.baseline_iea_comparison.shape)
print("top-level baseline.pkl:", baseline.shape)
print()
print("baseline.pkl first 5 columns :", baseline.columns[:5].tolist())
print("baseline.pkl last  3 columns :", baseline.columns[-3:].tolist())

## 4. Two long/stacked frames of run outputs

- `result.iea_comparison` — SISEPUEDE vs IEA at the crosswalk grain: `run_index * year * (iea_balance_code, iea_product_code)`. Handy for diagnostics but **not** what the surrogate learns from directly.
- `result.model_outputs` — raw SSP outputs (all fields, all years, all runs). This is where `build_training_matrices` reads Y from.


In [ ]:
print("iea_comparison columns:")
print(result.iea_comparison.columns.tolist())
print()
# Peek at one (run, year) slice as a wide table.
year0 = int(metadata["target_year"])
one_run = (
    result.iea_comparison
    .query("run_index == 0 and year == @year0")
    [["iea_balance_code", "iea_product_code",
      "value_sisepuede_tj", "value_iea_tj", "rel_err"]]
    .head(10)
)
print(f"iea_comparison @ run=0, year={year0} (first 10 rows):")
one_run

In [ ]:
print("model_outputs shape:", result.model_outputs.shape)
# What we'll actually pull as Y later on (raw per-tech SSP fields).
prod_cols = [
    c for c in result.model_outputs.columns
    if c.startswith("nemomod_entc_annual_production_by_technology_pp_")
]
print("# per-tech production columns:", len(prod_cols))
print("first 3:", prod_cols[:3])

## 5. Provenance in `metadata.json`

Everything downstream trusts these fields.

In [ ]:
for k in ["iso_country", "target_year", "seed", "n_lhs", "n_specs",
         "knob_bounds", "knob_prefix_filters",
         "consumption_fingerprint", "git_commit", "python_version",
         "generated_at"]:
    if k in metadata:
        print(f"  {k:26s} {metadata[k]}")
print()
print("first 4 spec_columns:")
for c in metadata["spec_columns"][:4]:
    print("   ", c)

## 6. Build the crosswalk and the aggregation matrix `A`

The surrogate is trained on **raw per-tech SSP fields**. At inference time we aggregate its predictions to IEA cells via a sparse `A`:

$$ y_\text{iea} = A \cdot y_\text{ssp}, \qquad A \in \mathbb{R}^{T_\text{iea} \times T_\text{ssp}}. $$

`crosswalk_ssp_columns_for_iea_targets` returns the union of SSP fields required by the requested IEA rows plus that matrix.

In [ ]:
from sisepuede.manager.sisepuede_file_structure import SISEPUEDEFileStructure
from sisepuede.calibration.iea_crosswalk import IEACrosswalk
from sisepuede.calibration._training_set import (
    crosswalk_ssp_columns_for_iea_targets,
    build_training_matrices,
    split_train_dev_test,
)
from sisepuede.calibration._training_pipeline import DEFAULT_V3_IEA_TARGETS

file_structure = SISEPUEDEFileStructure()
model_attrs = file_structure.model_attributes
crosswalk = IEACrosswalk(
    model_attrs,
    path_crosswalk=str(REPO_ROOT / "sisepuede/ref/data_crosswalks/sisepuede_iea_energy_crosswalk.csv"),
)

iea_target_rows = list(DEFAULT_V3_IEA_TARGETS)
print(f"IEA target rows ({len(iea_target_rows)}):")
for row in iea_target_rows:
    print("  ", row)

ssp_columns, A = crosswalk_ssp_columns_for_iea_targets(iea_target_rows, crosswalk)
print("\nssp_columns:", len(ssp_columns))
print("A shape    :", A.shape)
print("A first row:", A[0])

## 7. Build the training matrices `(X, Y)`

- `X` is `result.input_samples` renamed (identical values, same order).
- `Y` is `result.model_outputs` filtered to `target_year`, pivoted so each row is a run and each column is one of `ssp_columns`.

Both DataFrames share `run_index` as the row index.

In [ ]:
target_year = int(metadata["target_year"])
X, Y = build_training_matrices(result, target_year, ssp_columns)
print("X shape:", X.shape, " Y shape:", Y.shape)
print("X.index == Y.index:", (X.index == Y.index).all())
Y.head()

## 8. Split train / dev / test

Deterministic split at `split_seed`. Any real bundle (`n_lhs ≥ ~20`) restores a non-empty test split; on the n=2000 bundle it will be `1400 / 300 / 300`.

In [ ]:
splits = split_train_dev_test(X, Y, fractions=(0.7, 0.15, 0.15), seed=42)
(X_tr, Y_tr), (X_dv, Y_dv), (X_ts, Y_ts) = splits["train"], splits["dev"], splits["test"]
for name, (Xs, Ys) in splits.items():
    print(f"  {name:5s} X={Xs.shape}  Y={Ys.shape}")

## 9. `SurrogateSpec` — the hyperparameters that matter

The `SurrogateSpec` dataclass carries both the ML backend config **and** the accuracy gate thresholds. The four fields we'll iterate on most:

- `model_kind`: `"gbm"` by default (`HistGradientBoostingRegressor`, one per target).
- `hyperparams`: forwarded to the regressor. Sensible defaults live in `_default_hyperparams_for(model_kind)`.
- `target_r2_min`, `target_mape_max`: the accuracy-gate cut-offs applied to the TEST report. A target is accepted only if `R$^2$ ≥ target_r2_min` and (`MAPE` is NaN OR `MAPE ≤ target_mape_max`).
- `fail_mode`: `"warn"` logs rejections and continues; `"raise"` aborts if anything gets rejected.

In [ ]:
from sisepuede.calibration._surrogate import (
    Surrogate, SurrogateSpec, SurrogateReport, apply_accuracy_gate,
)

spec = SurrogateSpec(
    model_kind      = "gbm",
    seed            = 42,
    target_r2_min   = 0.85,
    target_mape_max = 20.0,
    fail_mode       = "warn",
)
spec

## 10. Train the surrogate

`Surrogate.train` fits one regressor per column of `Y`. If a target's `Y` is a constant, a `_ConstantRegressor` is substituted (auto-accepted by the gate). The LHS box is passed as `envelope_bounds` so `in_envelope(f)` will later know where inference is trustworthy.

In [ ]:
knob_lb, knob_ub = metadata["knob_bounds"]
n_knobs = X.shape[1]
envelope_bounds = (
    np.full(n_knobs, float(knob_lb)),
    np.full(n_knobs, float(knob_ub)),
)

surrogate = Surrogate.train(
    X_tr, Y_tr, spec,
    consumption_fingerprint=metadata["consumption_fingerprint"],
    envelope_bounds=envelope_bounds,
)
print(surrogate)
print("n_train_samples :", surrogate._n_train_samples)
print("envelope head   :")
print(surrogate.envelope.head())

## 11. Evaluate on dev and test

`Surrogate.evaluate(X_eval, Y_eval)` returns a `SurrogateReport` with per-target R$^2$ and MAPE Series. It does **not** apply the accuracy gate — that's a separate call. Convention: use dev for hyperparameter tuning, run the gate on test exactly once.


In [ ]:
def _summarise(report: SurrogateReport, label: str) -> pd.DataFrame:
    df = pd.DataFrame({
        "r2":   report.r2_per_target,
        "mape": report.mape_per_target,
    })
    print(f"[{label}] n_train={report.n_train}  n_holdout={report.n_holdout}")
    return df

dev_report  = surrogate.evaluate(X_dv, Y_dv) if X_dv.shape[0] else None
test_report = surrogate.evaluate(X_ts, Y_ts) if X_ts.shape[0] else None

if dev_report is not None:
    display(_summarise(dev_report,  "dev"))
if test_report is not None:
    display(_summarise(test_report, "test"))

## 12. Apply the accuracy gate

`apply_accuracy_gate(surrogate, report, spec)` mutates the report in place and returns `(accepted, rejected)`. Rules:

1. Constant-output targets -> auto-accept (gradient is zero, prediction is exact).
2. `n_holdout == 0` -> auto-accept (no defensible reject signal on in-sample scores).
3. Otherwise: accept iff `R$^2$ ≥ target_r2_min` and (`MAPE` NaN or `MAPE ≤ target_mape_max`).

In [ ]:
if test_report is None:
    print("no test partition available — accuracy gate skipped (n_test = 0)")
    print("tip: fall back to the dev report by calling apply_accuracy_gate(surrogate, dev_report, spec)")
else:
    accepted, rejected = apply_accuracy_gate(surrogate, test_report, spec)
    print("accepted (", len(accepted), "):")
    for t in accepted: print("  ", t)
    print("rejected (", len(rejected), "):")
    for t in rejected: print("  ", t)

## 13. Persist the surrogate bundle

`Surrogate.save(path)` writes a joblib payload — a plain dict wrapping the fitted estimators, the envelope, the spec, and the fingerprint. To keep the training payload independent from the aggregation, `A` and its labels are written to a separate `aggregation.pkl` by the higher-level `_persist_artifacts` helper (used by both `train_surrogate.py` and `train_surrogate_from_data`).


In [ ]:
workshop_dir = SURROGATE_OUT_DIR / f"{metadata['iso_country']}_{target_year}_workshop"
workshop_dir.mkdir(parents=True, exist_ok=True)

surrogate.save(str(workshop_dir / "surrogate.joblib"))
joblib.dump({"iea_target_rows": iea_target_rows, "ssp_columns": ssp_columns, "A": A},
            workshop_dir / "aggregation.pkl")

# Peek at the raw joblib payload — shows exactly what was serialised.
payload = joblib.load(workshop_dir / "surrogate.joblib")
print("joblib payload keys:", list(payload.keys()))
print("  columns  :", len(payload["columns"]), "knobs")
print("  targets  :", len(payload["targets"]), "IEA-ssp targets")
print("  spec     :", payload["spec"])
print("  envelope :", type(payload["envelope"]).__name__, payload["envelope"].shape)

In [ ]:
# Round-trip: load and use.
sur2 = Surrogate.load(str(workshop_dir / "surrogate.joblib"))
print(sur2)
print("predict at baseline (all ones):")
print(sur2.predict(sur2.baseline_inputs[None, :])[0])

## 14. One-shot: the same lifecycle via `train_surrogate_from_data`

Steps 6–13 above are what `train_surrogate_from_data` (the driver behind `train_surrogate.py`) does end-to-end. Use it once you've finished exploring.

In [ ]:
from sisepuede.calibration._training_pipeline import (
    train_surrogate_from_data, load_surrogate_bundle,
)

out = train_surrogate_from_data(
    training_data_dir = str(BUNDLE_DIR),
    crosswalk         = crosswalk,
    output_dir        = str(SURROGATE_OUT_DIR),
    tag               = "workshop_full",
    verbose           = True,
)
print()
print("keys returned :", list(out.keys()))
print("persisted at  :", out.get("output_dir"))

In [ ]:
reloaded = load_surrogate_bundle(out["output_dir"])
print("reloaded keys:", list(reloaded.keys()))
print(reloaded["surrogate"])

## Next steps

- **Run a large LHS bundle.** `python -m sisepuede.calibration.generate_surrogate_data --country PER --target-year 2018 --n-lhs N000 --seed 42 --knob-lb 0.9 --knob-ub 1.1 --calibrated-input /Users/dianamendez/sisepuede-data/input_data_per_calibrated_2018.csv` — then re-point `BUNDLE_DIR` at the resulting `PER_2018_nN000_seed42/`.
- **Iterate on `SurrogateSpec.hyperparams`.** Use `Surrogate.train` / `Surrogate.evaluate` on the dev split until you like the numbers, then call `apply_accuracy_gate` on the test split exactly once.
- **Feed the accepted surrogate to `ProductionCalibrator`** via `run_energy_calibration --cal-option 5 --surrogate <path>`.


## Appendix A — Overriding the default hyperparameters

The `SurrogateSpec.hyperparams` dict is **shallow-merged on top of the backend defaults** at fit time. Concretely, `_build_estimator` does:

```python
hparams = {**_default_hyperparams_for(spec.model_kind), **spec.hyperparams}
```

So you only pass the keys you want to override. Everything you don't pass keeps the default from `_default_hyperparams_for(model_kind)` in `_surrogate.py`:

| backend | key defaults (see `_default_hyperparams_for`) |
|---|---|
| `"gbm"`  | `max_iter=200`, `max_depth=4`, `learning_rate=0.08`, `min_samples_leaf=5`, `l2_regularization=1e-2` |
| `"xgb"`  | `n_estimators=200`, `max_depth=4`, `learning_rate=0.08`, `min_child_weight=5`, `reg_lambda=1e-2`, `tree_method="hist"` |
| `"lgbm"` | `n_estimators=200`, `num_leaves=15`, `learning_rate=0.08`, `min_data_in_leaf=5`, `reg_lambda=1e-2` |

`spec.seed` is passed as the regressor's `random_state` separately — you don't put it in `hyperparams`.

In [ ]:
# Concrete example: override GBM defaults with a deeper, more regularised model.
from sisepuede.calibration._surrogate import (
    SurrogateSpec, _default_hyperparams_for, _build_estimator,
)

spec_custom = SurrogateSpec(
    model_kind = "gbm",
    seed       = 42,
    hyperparams = {
        # Only the keys you list override the defaults; everything else is
        # inherited from _default_hyperparams_for("gbm").
        "max_iter":         500,      # more boosting rounds
        "max_depth":        6,        # deeper trees
        "learning_rate":    0.03,     # smaller step
        "l2_regularization": 0.1,     # stronger L2
    },
    target_r2_min   = 0.85,
    target_mape_max = 20.0,
)

print("defaults for gbm       :", _default_hyperparams_for("gbm"))
print("user override          :", spec_custom.hyperparams)

# Instantiate one estimator so we can verify what actually lands on the
# regressor (defaults + overrides + random_state).
est = _build_estimator(spec_custom, target_label=("ELECTOUT", "COAL"))
print("\nresulting estimator:", type(est).__name__)
for k in ["max_iter", "max_depth", "learning_rate",
          "min_samples_leaf", "l2_regularization", "random_state"]:
    print(f"  {k:20s} = {getattr(est, k)}")

# Train + evaluate the custom-hyperparameter surrogate on the existing splits
# to see if the numbers change vs. the default spec used earlier in the workshop.
sur_custom = Surrogate.train(
    X_tr, Y_tr, spec_custom,
    consumption_fingerprint = metadata["consumption_fingerprint"],
    envelope_bounds         = envelope_bounds,
)
dev_report_custom = sur_custom.evaluate(X_dv, Y_dv)
print("\ncustom-hyperparam dev R^2 (non-NaN):")
print(dev_report_custom.r2_per_target.dropna().describe())

## Appendix B — Automated hyperparameter search

`_hyperparam_search.py` sweeps combinations, trains one surrogate per combination on `(X_tr, Y_tr)`, evaluates on `(X_dv, Y_dv)`, and returns a ranked DataFrame plus a ready-to-use best `SurrogateSpec`. **The test split stays untouched** so the final `apply_accuracy_gate(surrogate, test_report, spec)` remains a real generalisation check.

- `grid_search_hyperparams(...)` — exhaustive Cartesian product.
- `random_search_hyperparams(..., n_iter=K)` — samples `K` combinations from the distribution dicts; use when the grid is too large to enumerate.
- Ranking metric via `metric=`: `"mean_r2"` / `"median_r2"` (maximised) or `"mean_mape"` / `"median_mape"` (minimised). Constant-output targets and NaN-MAPE targets are excluded from the aggregate — they auto-pass the gate anyway.

Once you have a `best_spec`, retrain on **train + dev** and only then apply the accuracy gate on test.

In [ ]:
from sisepuede.calibration._hyperparam_search import grid_search_hyperparams

# Small grid so this cell runs in ~a few minutes on the n=1000 workshop bundle.
# For a larger n=N000 bundle bump the ranges wider and pick metric="median_r2"
# (more robust to a single ill-behaved target dragging the mean).
param_grid = {
    "max_iter":         [200, 400],
    "max_depth":        [4, 6],
    "learning_rate":    [0.05, 0.08],
    "min_samples_leaf": [3, 5],
}

base_spec = SurrogateSpec(
    model_kind      = "gbm",
    seed            = 42,
    target_r2_min   = 0.85,
    target_mape_max = 20.0,
)

def _log(i, n, params, scores):
    print(f"  [{i:>2d}/{n}] mean_r2={scores.get('mean_r2', float('nan')):.4f}  "
          f"mean_mape={scores.get('mean_mape', float('nan')):.2f}%  "
          f"params={params}")

search = grid_search_hyperparams(
    X_train                 = X_tr,
    Y_train                 = Y_tr,
    X_dev                   = X_dv,
    Y_dev                   = Y_dv,
    base_spec               = base_spec,
    param_grid              = param_grid,
    envelope_bounds         = envelope_bounds,
    consumption_fingerprint = metadata["consumption_fingerprint"],
    metric                  = "mean_r2",
    on_progress             = _log,
)

print("\nranked results (top 5):")
display(search.results_df.head())
print("\nbest params :", search.best_params)
print("best score  :", search.best_score, "(", search.metric, ")")
print("best_spec   :", search.best_spec)